# Scenario catalog

20 representative driving scenarios from
`meta/scenarios_and_queries_reference.md` encoded as DSL queries.
Each row shows the candidate-clip count and an example matching clip id.


In [ ]:
from dataclasses import dataclass
from pathlib import Path
import pandas as pd

from causal_ai_av.dataset import CausalAVDataset

ds = CausalAVDataset(Path("/home/horde/01_json_annotations"))

@dataclass(frozen=True)
class Scenario:
    id: str
    title: str
    query: str


In [ ]:
SCENARIOS = [
    Scenario("1",  "Ego drives through crosswalk while pedestrian present",
             "agent.type = ped and env.type = crosswalk and ego.action = drive"),
    Scenario("2",  "Ego yields/stops at crosswalk for a pedestrian",
             "agent.type = ped and env.type = crosswalk and ego.action in (stop, yield, decel)"),
    Scenario("8",  "Pedestrian jaywalks; ego brakes / yields",
             "agent(type = ped, action(jaywalk = true)) and ego.action in (stop, yield, decel)"),
    Scenario("9",  "Pedestrian crosses while ego is turning",
             "agent.type = ped and env.type = crosswalk and ego.action in (turn_left, turn_right)"),
    Scenario("12", "Cyclist mid-block in ego's path; ego defensive",
             "agent.type = cyclist and ego.action in (stop, yield, decel)"),
    Scenario("14", "Ego stops at a red traffic light (nominal)",
             "light.color = red and ego.action = stop and env.type = intersection"),
    Scenario("16", "Signal blackout — ego treats as all-way stop",
             "light.state = off and ego.action = stop and env.type = intersection"),
    Scenario("20", "Ego already in intersection when yellow begins, proceeds",
             "light(color = yellow, ego_in_on_yellow = true) and ego.action in (drive, enter, creep)"),
    Scenario("21", "Ego approaches intersection, yellow appears, ego stops",
             "light(color = yellow, on_ego_path = true) and ego.action = stop"),
    Scenario("23", "Ego stops on yellow it could have cleared safely",
             "light(color = yellow, could_have_cleared = true) and ego.action = stop"),
    Scenario("aws-basic", "Ego stops at a stop sign",
             "obj.type = stop_sign and ego.action = stop"),
    Scenario("yield-basic", "Ego yields at a yield sign",
             "obj.type = yield_sign and ego.action = yield"),
    Scenario("76", "Vehicle stopped in front of ego, ego nudges or changes lane",
             "agent(type = vehicle, pos = front, action(type in (stop, not_move))) "
             "and ego.action in (nudge, change_lane_left, change_lane_right)"),
    Scenario("lane-multi", "Multi-lane road, ego changes lane",
             "env(type = road, lanes >= 2) and ego.action in (change_lane_left, change_lane_right)"),
    Scenario("officer-stop", "Officer signaling stop; ego stops",
             "agent(type = officer, signaling = stop) and ego.action = stop"),
    Scenario("144", "Ego transits roundabout without yielding",
             "env.type = roundabout and ego.action = drive"),
    Scenario("142", "Vehicle cuts in at roundabout exit; ego brakes",
             "env.type = roundabout "
             "and agent(type = vehicle, action(type in (change_lane, change_lane_left, change_lane_right))) "
             "and ego.action in (decel, stop, nudge)"),
    Scenario("decel-because-ped", "Ego decelerates BECAUSE OF a pedestrian (causal edge)",
             "ego.action = decel because_of agent.type = ped"),
    Scenario("yellow-then-stop", "Yellow light followed by ego stop within 3s",
             "light.color = yellow then(3) ego.action = stop"),
    Scenario("red-without-stop", "During red light, ego never stops (potential violation)",
             "within light.color = red: not ego.action = stop"),
]
len(SCENARIOS)


## Run them all and tabulate


In [ ]:
rows = []
for s in SCENARIOS:
    try:
        matches = ds.find(s.query)
        clips = list(set(matches.clips()))
        rows.append({
            "id": s.id,
            "title": s.title,
            "candidates": len(clips),
            "example": clips[0] if clips else "",
            "query": s.query,
        })
    except Exception as e:
        rows.append({"id": s.id, "title": s.title, "candidates": -1,
                     "example": f"ERROR: {e}", "query": s.query})

df = pd.DataFrame(rows).sort_values("candidates", ascending=False)
df[["id", "title", "candidates", "example"]]


## Dig into one

Pick a scenario, list every matching clip.


In [ ]:
target = "agent(type = ped, action(jaywalk = true)) and ego.action in (stop, yield, decel)"
clips = sorted(set(ds.find(target).clips()))
print(f"{len(clips)} clips for: {target}")
for c in clips[:10]:
    print(" ", c)
